In [4]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV, PredefinedSplit
import joblib

In [6]:
df_base = pd.read_csv("baseline_data_cleaned.csv", index_col="oid", parse_dates=["veto_date"])
folds = pd.read_csv("stratified_folds.csv", index_col="oid")

In [7]:
features = [
    "n_points_alert","mag_min_alert","mag_max_alert",
    "mag_p05_alert","mag_p25_alert","mag_p50_alert","mag_p75_alert","mag_p95_alert",
    "excess_variance_alert","n_points_dr","mag_min_dr","mag_max_dr",
    "mag_p05_dr","mag_p25_dr","mag_p50_dr","mag_p75_dr","mag_p95_dr",
    "excess_variance_dr","distance_delight","hostsize"
]
X = df_base[features]
y = df_base["label"]

In [8]:
# Construimos los folds de validacion a partid del csv predefinido
test_fold = np.full(len(df_base), -1, dtype=int)
for i in range(1,6):
    mask = folds[f"val_fold{i}"] == 1
    test_fold[mask.values] = i-1

In [9]:
ps = PredefinedSplit(test_fold=test_fold)

In [11]:
# GridSeach parametros
param_grid = {
    "n_estimators":      [100, 200, 500],
    "max_depth":         [None, 10, 20],
    "min_samples_split": [2, 5, 10],
    "max_features":      ["sqrt", "log2"]
}

In [12]:
# Modelo RandomForest
rf = RandomForestClassifier(random_state=42, n_jobs=-1)

In [13]:
# GridSearchCV
grid = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=ps,
    scoring="accuracy",
    verbose=2
)

In [14]:
# Buscamos hyperparametros
grid.fit(X, y)

print("Mejores parámetros encontrados:")
print(grid.best_params_)
print("Mejor accuracy (CV):", grid.best_score_)

Fitting 5 folds for each of 54 candidates, totalling 270 fits
[CV] END max_depth=None, max_features=sqrt, min_samples_split=2, n_estimators=100; total time=   2.2s
[CV] END max_depth=None, max_features=sqrt, min_samples_split=2, n_estimators=100; total time=   2.3s
[CV] END max_depth=None, max_features=sqrt, min_samples_split=2, n_estimators=100; total time=   2.0s
[CV] END max_depth=None, max_features=sqrt, min_samples_split=2, n_estimators=100; total time=   2.1s
[CV] END max_depth=None, max_features=sqrt, min_samples_split=2, n_estimators=100; total time=   2.0s
[CV] END max_depth=None, max_features=sqrt, min_samples_split=2, n_estimators=200; total time=   4.3s
[CV] END max_depth=None, max_features=sqrt, min_samples_split=2, n_estimators=200; total time=   4.1s
[CV] END max_depth=None, max_features=sqrt, min_samples_split=2, n_estimators=200; total time=   4.2s
[CV] END max_depth=None, max_features=sqrt, min_samples_split=2, n_estimators=200; total time=   4.2s
[CV] END max_depth=N

KeyboardInterrupt: 

In [ ]:
# Evaluamos sobre cada fold
best_rf = grid.best_estimator_
for i in range(1,6):
    train_mask = folds[f"train_fold{i}"] == 1
    val_mask   = folds[f"val_fold{i}"]   == 1

    X_train, y_train = X[train_mask.values], y[train_mask.values]
    X_val,   y_val   = X[val_mask.values],   y[val_mask.values]

    y_pred = best_rf.fit(X_train, y_train).predict(X_val)
    print(f"\n=== Fold {i} ===")
    print(classification_report(y_val, y_pred))